# tamserve: four cameras, one GPU

Tracks objects on four videos at once, played as cameras, with one copy of EdgeTAM. Runtime → Change runtime type → a GPU (the free T4 works), then Runtime → Run all.

In [ ]:
!pip install -q tamserve opencv-python-headless
import torch
print(torch.cuda.get_device_name())

Four 10-second clips from [Pexels](https://www.pexels.com), cut to 1280 x 720 and attached to the tamserve release; free to use under the [Pexels licence](https://www.pexels.com/license/). Originals: [welder](https://www.pexels.com/video/14314645/), [cat](https://www.pexels.com/video/7727397/), [airport](https://www.pexels.com/video/856984/), [office](https://www.pexels.com/video/6914684/). The boxes say where each object is in the first frame; in a real system they come from a detector or a click.

In [ ]:
import urllib.request
import cv2
import numpy as np

CLIPS = {  # Pexels id -> {object: (left, top, right, bottom) on the first frame}
    "14314645": {"welder": (228, 138, 787, 714)},
    "7727397": {"cat": (620, 212, 1052, 713)},
    "856984": {"plane": (536, 309, 799, 410), "marshaller": (1073, 571, 1120, 631)},
    "6914684": {"person 1": (3, 282, 388, 631), "person 2": (480, 190, 736, 558), "person 3": (877, 186, 1279, 573)},
}
BASE = "https://github.com/aryamandarda/tamserve/releases/download/v0.2.0/"


def frames_of(pexels_id):
    """The clip's frames as 1280 x 720 RGB arrays, as a camera would send them."""
    path = f"/tmp/{pexels_id}.mp4"
    urllib.request.urlretrieve(f"{BASE}{pexels_id}.mp4", path)
    capture, frames = cv2.VideoCapture(path), []
    while (frame := capture.read())[0]:
        frames.append(cv2.cvtColor(frame[1], cv2.COLOR_BGR2RGB))
    return frames


videos = {pexels_id: frames_of(pexels_id) for pexels_id in CLIPS}
print({pexels_id: len(frames) for pexels_id, frames in videos.items()})

One engine, one call per round: each round hands it the next frame of every camera and gets back every object's mask.

In [ ]:
import time
from tamserve import Box, Engine, build_video_predictor

engine = Engine(build_video_predictor("cuda"))  # EdgeTAM. Engine(..., compile=True) is faster, after a long first compile.
masks = {pexels_id: [] for pexels_id in videos}
start = time.perf_counter()
for k in range(300):
    frames = {pexels_id: v[k] for pexels_id, v in videos.items() if k < len(v)}
    prompts = {pexels_id: {name: Box(*box) for name, box in CLIPS[pexels_id].items()} for pexels_id in frames} if k == 0 else None
    for pexels_id, objects in engine.step(frames, prompts).items():
        masks[pexels_id].append({name: m[::2, ::2].numpy() for name, m in objects.items()})  # half size, for drawing
took = time.perf_counter() - start
n = sum(len(m) for m in masks.values())
print(f"{n} frames from {len(videos)} cameras, {sum(map(len, CLIPS.values()))} objects, in {took:.1f} s: "
      f"{n / took:.0f} frames/s on {torch.cuda.get_device_name()}")

In [ ]:
import subprocess
from IPython.display import Video

COLORS = [(255, 99, 71), (65, 160, 255), (255, 200, 40)]
writer = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", "1280x720",
                           "-r", "30", "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", "/tmp/wall.mp4"],
                          stdin=subprocess.PIPE)
for k in range(300):
    wall = np.zeros((720, 1280, 3), np.uint8)
    for t, pexels_id in enumerate(videos):
        if k >= len(masks[pexels_id]):
            continue
        tile = cv2.resize(videos[pexels_id][k], (640, 360), interpolation=cv2.INTER_AREA).astype(np.float32)
        for j, m in enumerate(masks[pexels_id][k].values()):
            m = cv2.resize(m.astype(np.uint8), (640, 360), interpolation=cv2.INTER_NEAREST).astype(bool)
            tile[m] = tile[m] * 0.5 + np.array(COLORS[j % 3]) * 0.5
        y, x = divmod(t, 2)
        wall[y * 360:(y + 1) * 360, x * 640:(x + 1) * 640] = tile.astype(np.uint8)
    writer.stdin.write(wall.tobytes())
writer.stdin.close()
writer.wait()
Video("/tmp/wall.mp4", embed=True, width=960)

Next: live cameras, several GPUs and machines, and what happens when a camera drops, in the [README](https://github.com/aryamandarda/tamserve#using-it).